### Implementasi VSM

#### Import Library

In [ ]:
pip install Sastrawi nltk # menginstall sastrawi dan nltk

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import nltk
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

#### Import Data

In [8]:
df = pd.read_csv('Sindonews.csv')
df = df[['Judul', 'Isi Berita', "Kategori"]]
df.head()

,Judul,Isi Berita,Kategori
0,Kejagung Beberkan Peran Tom Lembong di Kasus I...,JAKARTA - Kejaksaan Agung (Kejagung) menetapka...,nasional
1,"Prabowo Larang Menteri Pakai Mobil Impor, Mobi...",JAKARTA - Presiden Prabowo Subianto meminta ...,nasional
2,"Instruksikan Menteri Gunakan Maung, Logis 08: ...",JAKARTA - Presiden terpilih Prabowo Subianto k...,nasional
3,"7 Fakta Ipda Rudy Soik, Perwira Polisi yang Di...","JAKARTA - Ipda Rudy Soik , Anggota Polda Nusa...",nasional
4,Bendum Perindo Angkie Yudistia Terima Pengharg...,JAKARTA - Bendahara Umum (Bendum) Partai Perin...,nasional


Meng import data hasil crawling dari website **SindoNews** yang terdiri dari 200 Data yang berisi Judul, Isi Berita, Tanggal Berita dan Kategori.

In [9]:
# Filter data sebanyak 50 berdasarkan kategori
Nasional = df[df["Kategori"] == "nasional"].sample(n=50, random_state=42)
Internasional = df[df["Kategori"] == "internasional"].sample(n=50, random_state=42)

In [ ]:
# Gabungkan 2 Kategori
data = pd.concat([Nasional, Internasional], ignore_index=True)
data.head()

,Judul,Isi Berita,Kategori
0,Ketum Partai Perindo Angela Tanoesoedibjo Ajak...,JAKARTA - Ketua Umum Partai Perindo Angela T...,nasional
1,Bendum Perindo Angkie Yudistia Terima Pengharg...,JAKARTA - Bendahara Umum (Bendum) Partai Perin...,nasional
2,"3 Menlu RI Sebelum Sugiono, Retno Marsudi 10 T...",JAKARTA - Tiga Menteri Luar Negeri ( Menlu ) R...,nasional
3,"Profil Letkol Laut Romi Habe Putra, Ajudan Pre...",JAKARTA - Profil Letkol Laut (P) Romi Habe Pu...,nasional
4,Wamentan Sudaryono Dorong Sumber Protein Alter...,JAKARTA - Pemerintah Indonesia mempersiapkan p...,nasional


#### Case Folding

**Case Folding** merupakan proses dalam prepocessing text untuk mengubah karakter dalam teks menjadi huruf kecil (lowercase). Tujuannya untuk menyamakan format teks.

In [ ]:
data = data.apply(lambda x: x.astype(str).str.lower())

In [ ]:
data.head()

,Judul,Isi Berita,Kategori
0,hizbullah luncurkan gelombang baru serangan ke...,beirut - hizbullah lebanon mengumumkan para pe...,internasional
1,"kejati jatim tangkap ronald tannur, kejagung: ...",jakarta - kejaksaan tinggi (kejati) jawa timur...,nasional
2,"kolaborasi dengan bumn pangan, mentan amran op...",jakarta - menteri pertanian (mentan) andi amra...,nasional
3,"tom lembong tersangka kasus impor gula, intip ...",jakarta - mantan menteri perdagangan (mendag) ...,nasional
4,gmni yakin kabinet merah putih mampu jalankan ...,jakarta - gerakan mahasiswa nasional indonesia...,nasional


In [ ]:
# Mengacak Data
data = data.sample(frac = 1, ignore_index=True)

In [ ]:
data.head()

,Judul,Isi Berita,Kategori
0,Hizbullah Luncurkan Gelombang Baru Serangan ke...,BEIRUT - Hizbullah Lebanon mengumumkan para pe...,internasional
1,"Kejati Jatim Tangkap Ronald Tannur, Kejagung: ...",JAKARTA - Kejaksaan Tinggi (Kejati) Jawa Timur...,nasional
2,"Kolaborasi dengan BUMN Pangan, Mentan Amran Op...",JAKARTA - Menteri Pertanian (Mentan) Andi Amra...,nasional
3,"Tom Lembong Tersangka Kasus Impor Gula, Intip ...",JAKARTA - Mantan Menteri Perdagangan (Mendag) ...,nasional
4,GMNI Yakin Kabinet Merah Putih Mampu Jalankan ...,JAKARTA - Gerakan Mahasiswa Nasional Indonesia...,nasional


In [ ]:
data.to_csv('Final_Data.csv', index=False)

Data yang sudah difilter berdasarkan kategori, masing masing kategori sebanyak 50 lalu data tersebut digabung lalu data tersebut diacak

#### Tokenizing

**Tokenizing** adalah proses dalam Prepocessing Text yang bertujuan untuk memisahkan atau memecah teks menjadi bagian–bagian kata yang disebut token.

In [ ]:
Tokenizing = pd.DataFrame()

# Terapkan tokenisasi pada kolom 'Judul' dan 'Isi Berita'
Tokenizing['Judul'] = data_bersih['Judul'].apply(word_tokenize)
Tokenizing['Isi'] = data_bersih['Isi Berita'].apply(word_tokenize)
Tokenizing['Kategori'] = data_bersih['Kategori']

# Menampilkan hasil tokenisasi
Tokens = Tokenizing[['Judul', 'Isi','Kategori']]

In [ ]:
Tokens.head()

,Judul,Isi,Kategori
0,"[hizbullah, luncurkan, gelombang, baru, serang...","[beirut, hizbullah, lebanon, mengumumkan, para...",internasional
1,"[kejati, jatim, tangkap, ronald, tannur, kejag...","[jakarta, kejaksaan, tinggi, kejati, jawa, tim...",nasional
2,"[kolaborasi, dengan, bumn, pangan, mentan, amr...","[jakarta, menteri, pertanian, mentan, andi, am...",nasional
3,"[tom, lembong, tersangka, kasus, impor, gula, ...","[jakarta, mantan, menteri, perdagangan, mendag...",nasional
4,"[gmni, yakin, kabinet, merah, putih, mampu, ja...","[jakarta, gerakan, mahasiswa, nasional, indone...",nasional


#### Filtering

**Filtering** merupakan proses dalam Prepocessing Text untuk pemilihan kata-kata penting dari hasil tokenizing, yaitu kata-kata yang bisa digunakan untuk mewakili isi dari sebuah teks atau dokumen. Proses filtering juga biasa disebut sebagai stopword removal. Filtering adalah proses penyaringan kata-kata atau elemen dari teks berdasarkan kriteria tertentu. Biasanya, ini dilakukan untuk menghapus kata-kata yang tidak relevan atau tidak memberikan informasi penting dalam analisis teks.

In [ ]:
stop_words = set(stopwords.words('indonesian'))

# Menyimpan stopwords ke dalam file stopwords.txt
with open('stopwords.txt', 'w') as f:
    for item in stop_words:
        f.write("%s\n" % item)

In [ ]:
def clean_stopword(tokens):
	listStopword =  set(stopwords.words('indonesian'))
	removed = []
	for i in tokens:
		if i not in listStopword:
			removed.append(i)
	return removed

In [ ]:
# Menghapus stopwords dari kolom 'Judul' dan 'Isi'
Tokens['Judul_cleaned'] = Tokens['Judul'].apply(clean_stopword)
Tokens['Isi_cleaned'] = Tokens['Isi'].apply(clean_stopword)

In [ ]:
# Menampilkan hasil akhir
StopWord = Tokens[['Judul_cleaned', 'Isi_cleaned', 'Kategori']]
StopWord.head()

,Judul_cleaned,Isi_cleaned,Kategori
0,"[hizbullah, luncurkan, gelombang, serangan, is...","[beirut, hizbullah, lebanon, mengumumkan, peju...",internasional
1,"[kejati, jatim, tangkap, ronald, tannur, kejag...","[jakarta, kejaksaan, kejati, jawa, timur, mena...",nasional
2,"[kolaborasi, bumn, pangan, mentan, amran, opti...","[jakarta, menteri, pertanian, mentan, andi, am...",nasional
3,"[tom, lembong, tersangka, impor, gula, intip, ...","[jakarta, mantan, menteri, perdagangan, mendag...",nasional
4,"[gmni, kabinet, merah, putih, jalankan, progra...","[jakarta, gerakan, mahasiswa, nasional, indone...",nasional


In [ ]:
StopWord.to_csv('StopWord_Sindonews.csv', index=False)

#### Stemming

Stemming adalah proses dalam Prepocessing Text yang bertujuan untuk mengurangi kata-kata ke bentuk dasarnya (stem).

In [ ]:
def stemming(text):
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    # Jika input adalah list, gabungkan menjadi string
    if isinstance(text, list):
        text = ' '.join(text)

    # Lakukan stemming
    text = ' '.join(stemmer.stem(word) for word in text.split())
    return text

In [ ]:
stemming_df = pd.DataFrame()

# Terapkan stemming pada kolom 'Judul' dan 'Isi Berita'
stemming_df['Judul'] = StopWord['Judul_cleaned'].apply(stemming)
stemming_df['Isi'] = StopWord['Isi_cleaned'].apply(stemming)
stemming_df['Kategori'] = StopWord['Kategori']

# Menampilkan hasil stemming
stemming_df = stemming_df[['Judul', 'Isi', 'Kategori']]

In [ ]:
stemming_df

,Judul,Isi,Kategori
0,hizbullah luncur gelombang serang israel operasi,beirut hizbullah lebanon umum juang lawan isla...,internasional
1,kejat jatim tangkap ronald tannur jagung laks...,jakarta jaksa kejat jawa timur tangkap gregori...,nasional
2,kolaborasi bumn pangan tan amran optimal mandi...,jakarta menteri tani tan andi amran sulaiman s...,nasional
3,tom lembong sangka impor gula intip harta kaya,jakarta mantan menteri dagang mendag thomas tr...,nasional
4,gmni kabinet merah putih jalan program prabowo,jakarta gera mahasiswa nasional indonesia gmni...,nasional
...,...,...,...
95,kerja rumah presiden prabowo jaga daulat bangsa,jakarta ketua dewan pimpin nasional asosiasi t...,nasional
96,malam abraham silaban ab presiden prabowo maun...,jakarta maung garuda mv garuda limousine pusat...,nasional
97,terima harga angkie yudistia kawal inklusivit...,jakarta staf khusus presiden ri periode angkie...,nasional
98,deputi kpk pahala nainggolan polda metro perik...,jakarta deputi cegah monitoring komisi beranta...,nasional


In [ ]:
stemming_df.to_csv('Stemming_Sindonews.csv', index=False)

#### Menghitung TF-IDF

TF-IDF (Term Frequency - Inverse Document Frequency) adalah teknik yang digunakan dalam pemrosesan teks untuk menilai pentingnya suatu kata dalam suatu dokumen relatif terhadap kumpulan dokumen (corpus).

Konsep ini terdiri dari dua bagian:


1.   Term Frequency (TF): Mengukur seberapa sering sebuah kata muncul dalam sebuah dokumen.
2.   Inverse Document Frequency (IDF): Mengukur seberapa jarang kata tersebut muncul dalam kumpulan dokumen (corpus).



Term Frequency (TF):

$$
\text{TF}(t, d) = \frac{\text{Jumlah kemunculan t dalam d}}{\text{Total jumlah kata dalam d}}
$$
Arti dari rumus ini:

1. TF(t, d): Menunjukkan frekuensi kata t dalam dokumen d.
2. Jumlah kemunculan t dalam d: Merupakan jumlah kali kata t muncul di dalam dokumen d.
3. Total jumlah kata dalam d: Merupakan total semua kata yang terdapat dalam dokumen d.


Inverse Document Frequency (IDF):

$$
\text{IDF}(t, D) = \log \left( \frac{N}{|\{d \in D : t \in d\}|} \right)
$$



TF-IDF:

$$
\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)
$$
TF-IDF(t, d, D):
1. Menunjukkan nilai TF-IDF untuk kata t dalam dokumen d dari kumpulan dokumen D. Ini adalah nilai yang mengindikasikan pentingnya kata tersebut dalam konteks dokumen dan corpus.
2. TF(t, d):Nilai Term Frequency untuk kata t dalam dokumen d. Ini mengukur seberapa sering kata t muncul dalam dokumen d. Semakin tinggi nilai TF, semakin sering kata tersebut muncul dalam dokumen, yang menunjukkan relevansinya.
3. IDF(t, D):Nilai Inverse Document Frequency untuk kata t dalam kumpulan dokumen D. Ini mengukur seberapa jarang kata tersebut muncul di seluruh corpus. Semakin jarang kata t muncul, semakin tinggi nilai IDF, yang menunjukkan bahwa kata tersebut lebih spesifik dan relevan.

In [ ]:
import pandas as pd
df_Stemming = pd.read_csv('Stemming_Sindonews.csv')
df_Stemming.head()

,Judul,Isi,Kategori
0,hizbullah luncur gelombang serang israel operasi,beirut hizbullah lebanon umum juang lawan isla...,internasional
1,kejat jatim tangkap ronald tannur jagung laks...,jakarta jaksa kejat jawa timur tangkap gregori...,nasional
2,kolaborasi bumn pangan tan amran optimal mandi...,jakarta menteri tani tan andi amran sulaiman s...,nasional
3,tom lembong sangka impor gula intip harta kaya,jakarta mantan menteri dagang mendag thomas tr...,nasional
4,gmni kabinet merah putih jalan program prabowo,jakarta gera mahasiswa nasional indonesia gmni...,nasional


In [ ]:
def TFIDF(df):
  vectorizer = TfidfVectorizer()
  # perhitungan TF-IDF
  tfidf_matrix = vectorizer.fit_transform(df)
  # Mendapatkan daftar fitur hasil TF-IDF
  feature_names = vectorizer.get_feature_names_out()
  # Mengonversi hasil TF-IDF ke dalam DataFrame
  tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
  return tfidf_df

In [ ]:
# Menghitung TF-IDF untuk kolom 'Isi'
TF_IDF_ISI = TFIDF(df_Stemming['Isi'])

# Menambahkan kolom 'Kategori' ke dalam hasil DataFrame TF-IDF
TF_IDF_ISI['Kategori'] = df_Stemming['Kategori']

In [ ]:
TF_IDF_ISI

,aal,ab,abadi,abai,abbas,abby,abd,abdi,abdul,abdulrahman,...,zarof,zibakalam,zik,zionis,znpp,zona,zourabichvili,zr,zulkifli,Kategori
0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.14924,0.0,0.0,0.0,0.0,0.0,internasional
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.051014,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
3,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
4,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
96,0.0,0.116315,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.058157,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
97,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
98,0.0,0.000000,0.0,0.0,0.0,0.0,0.042075,0.0,0.040673,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional


In [ ]:
TF_IDF_ISI.to_csv('TF_IDF_ISI.csv', index=False)

## Klasifikasi dengan Logistic Linear

salah satu teknik dalam statistik dan pembelajaran mesin yang digunakan untuk memodelkan hubungan antara variabel independen dan variabel dependen biner (dua kategori, misalnya "ya" atau "tidak"). Meskipun disebut "linear," regresi logistik tidak memodelkan hubungan linear antara variabel-variabel ini, melainkan menggunakan fungsi logistik atau sigmoid untuk menghasilkan probabilitas sebagai output.

### Import Library

In [ ]:
# Library untuk data manipulation & visualisasi
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Library untuk preprocessing data
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

# Library untuk model & evaluasi
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Library untuk save model
import pickle

### LoadData Set

In [ ]:
df = pd.read_csv('TF_IDF_ISI.csv')

df

,aal,ab,abadi,abai,abbas,abby,abd,abdi,abdul,abdulrahman,...,zarof,zibakalam,zik,zionis,znpp,zona,zourabichvili,zr,zulkifli,Kategori
0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.14924,0.0,0.0,0.0,0.0,0.0,internasional
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.051014,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
3,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
4,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
96,0.0,0.116315,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.058157,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
97,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional
98,0.0,0.000000,0.0,0.0,0.0,0.0,0.042075,0.0,0.040673,0.000000,...,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,nasional


### Label Encoding

Label encoding adalah teknik dalam praproses data yang digunakan untuk mengubah nilai-nilai kategoris (yang berupa teks atau string) menjadi bentuk numerik. Ini penting dalam banyak algoritma pembelajaran mesin yang hanya dapat bekerja dengan data numerik, bukan data kategoris.

In [ ]:
label_encoder = preprocessing.LabelEncoder()

# encode label
df['Kategori'] = label_encoder.fit_transform(df['Kategori'])

df.head()

,aal,ab,abadi,abai,abbas,abby,abd,abdi,abdul,abdulrahman,...,zarof,zibakalam,zik,zionis,znpp,zona,zourabichvili,zr,zulkifli,Kategori
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.14924,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.051014,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,1


### Pembagian Data Training dan Test

In [ ]:
X = df.drop(['Kategori'], axis=1)
y = df['Kategori']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Split data antara data latih dan data uji dengan perbandingan 80:20

### Model Regresion Linear

#### Training

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

#### Testing

In [ ]:
y_pred = model.predict(X_test)

y_pred

array([1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1])

#### Membandingkan Data

In [ ]:
Predict = pd.DataFrame({'Data asli':y_test, 'Data Prediksi':y_pred})
Predict.head()

,Data asli,Data Prediksi
83,1,1
53,1,1
70,1,1
45,1,1
44,0,0


#### Evaluasi Model

In [ ]:
# Evaluasi Model
accuracy = accuracy_score(y_test, y_pred)
print(f'Akurasi : {accuracy * 100}')

report = classification_report(y_test, y_pred)
print(report)

Akurasi : 95.0
              precision    recall  f1-score   support

           0       1.00      0.86      0.92         7
           1       0.93      1.00      0.96        13

    accuracy                           0.95        20
   macro avg       0.96      0.93      0.94        20
weighted avg       0.95      0.95      0.95        20



Dari hasil evaluasi diatas menunjukkan hasil akurasi skor sebesar 100 %

## Save model

In [ ]:
with open('lr_model.pkl', 'wb') as f:
    pickle.dump(model, f)